## My project becomes real SaaS engineering.

But we must adapt it carefully because you said:

👉 Run in Google Colab

👉 No paid API keys

So we will build a fully FREE SaaS-style AI system using:

- SQLite database (users + chats)
- Local open-source LLM (no API key)
- Flask API
- Ngrok public URL
- Simple login + multi-user chat

This will simulate a production SaaS architecture

## Big Picture — What we will build in Colab

Architecture we will create:

User (Browser)
→ Streamlit UI (Colab tunnel)

→ Flask API

→ SQLite Database

→ Local AI Model (FREE)

So yes — real SaaS flow

## STEP 0 — Install Everything (Colab)

Run this cell first:


In [ ]:
!pip install flask flask-cors pyngrok transformers accelerate sentencepiece sqlite-utils

## STEP 1 — Create Database (Users + Chats)

Create file db.py

You need to create the db.py file first before importing it. In Colab, you can do this by writing the file with a %%writefile magic:

In [4]:
%%writefile db.py
import sqlite3

def create_tables():
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    # Users table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        password TEXT
    )
    """)

    # Chats table (user specific)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chats(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        message TEXT,
        reply TEXT
    )
    """)

    conn.commit()
    conn.close()

Writing db.py


In [5]:
# Run once:

from db import create_tables
create_tables()
print("✔ Database ready.")

✔ Database ready.


## Summary

- The error occurs because the db.py file doesn’t exist yet.

- Fix by creating db.py with %%writefile in Colab.

- After that, the import will succeed and your SQLite tables will be created.

Database ready.

## STEP 2 — User Registration + Login Functions

Add in db.py

In [6]:
def register_user(username, password):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO users (username, password) VALUES (?, ?)",
        (username, password)
    )

    conn.commit()
    conn.close()


def login_user(username, password):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT * FROM users WHERE username=? AND password=?",
        (username, password)
    )

    result = cursor.fetchone()
    conn.close()
    return result


def save_chat(user_id, message, reply):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO chats (user_id, message, reply) VALUES (?, ?, ?)",
        (user_id, message, reply)
    )

    conn.commit()
    conn.close()

Now we have multi-user database

## STEP 3 — Add FREE Local AI Model (No API Key)

We use HuggingFace free model:

We need to use AutoModelForSeq2SeqLM instead of AutoModelForCausalLM:

### Key Difference

- Causal LM (AutoModelForCausalLM) → decoder‑only models (GPT family).

- Seq2Seq LM (AutoModelForSeq2SeqLM) → encoder‑decoder models (T5, BART, FLAN‑T5).

- FLAN‑T5 is instruction‑tuned T5, so you must use the Seq2Seq loader.

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_reply(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_reply("Translate English to French: Hello, how are you?"))

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Bonjour, c'est-à-dire?


### Summary

- Error occurs because you used the wrong model class.

- Fix by switching to AutoModelForSeq2SeqLM.

- Your FLAN‑T5 will then generate replies correctly.

This is your local ChatGPT replacement

## STEP 4 — Create Flask API (Backend)

Create app.py

In [11]:
%%writefile app.py
from flask import Flask, request, jsonify
from flask_cors import CORS
from db import register_user, login_user, save_chat
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

app = Flask(__name__)
CORS(app)

# Load model once (FLAN-T5 is seq2seq, not causal LM)
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_reply(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Register endpoint
@app.route("/register", methods=["POST"])
def register():
    data = request.json
    register_user(data["username"], data["password"])
    return jsonify({"status": "registered"})

# Login endpoint
@app.route("/login", methods=["POST"])
def login():
    data = request.json
    user = login_user(data["username"], data["password"])
    if user:
        return jsonify({"status": "success"})
    return jsonify({"status": "fail"})

# Chat endpoint
@app.route("/chat", methods=["POST"])
def chat():
    data = request.json
    user_id = data["user_id"]
    message = data["message"]

    reply = generate_reply(message)
    save_chat(user_id, message, reply)

    return jsonify({"response": reply})

if __name__ == "__main__":
    app.run(port=5000)

Overwriting app.py


Run it:

```
!python app.py
```

Flask is running locally

In [13]:
# Install dependencies:

!pip install flask flask-cors transformers accelerate sentencepiece sqlite-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 3.1 MB/s eta 0:00:00


### How to run Flask locally inside Colab and show your work
Install dependencies (if not already):

In [27]:
!pip install flask flask-cors transformers accelerate sentencepiece sqlite-utils

This will start the server on port 5000 inside Colab.

Expose it publicly (so you can access it in your browser):
Since ngrok now requires tokens, you can use Localtunnel or Cloudflared (both free, no account required):

### Option A — Localtunnel

In [31]:
!npm install -g localtunnel
!lt --port 5000

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧your url is: https://evil-pillows-arrive.loca.lt
API_URL = "https://giant-jobs-rest.loca.lt"
^C


This will print a public URL like:

```
https://randomname.loca.lt
```
Use this URL in your Streamlit frontend (API_URL).

### Option B — Cloudflared

In [ ]:
!pip install cloudflared
!cloudflared tunnel --url http://localhost:5000

This will print a public URL like:

```
https://something.trycloudflare.com
```
### Summary

- !python app.py runs Flask inside Colab, but you can’t access it directly.

- Use Localtunnel or Cloudflared to expose it without tokens.

- Replace API_URL in your Streamlit frontend with the public URL you get.

- Now you can demonstrate the full SaaS flow: Browser → Streamlit → Flask → SQLite → FLAN‑T5.

## STEP 5 — Make API Public (Ngrok)

Colab cannot expose localhost directly → use ngrok.

```
from pyngrok import ngrok
public_url = ngrok.connect(5000)
print(public_url)

```
You will get:

```
https://xxxx.ngrok-free.app
```
This is your public SaaS backend URL

## STEP 6 — Create Streamlit Frontend

Create streamlit_app.py

In [24]:

%%writefile frontend.py
import streamlit as st
import requests

API_URL = "PASTE_NGROK_OR_LOCALTUNNEL_URL_HERE"

st.title("AI SaaS Chat")

menu = ["Login", "Register"]
choice = st.sidebar.selectbox("Menu", menu)

username = st.text_input("Username")
password = st.text_input("Password", type="password")

if choice == "Register":
    if st.button("Register"):
        requests.post(API_URL + "/register",
                      json={"username": username, "password": password})
        st.success("Registered!")

if choice == "Login":
    if st.button("Login"):
        res = requests.post(API_URL + "/login",
                            json={"username": username, "password": password})
        if res.json()["status"] == "success":
            st.session_state["user"] = username
            st.success("Logged in!")

if "user" in st.session_state:
    st.write("Welcome", st.session_state["user"])

    msg = st.text_input("Ask AI")

    if st.button("Send"):
        res = requests.post(API_URL + "/chat",
                            json={"user_id": st.session_state["user"],
                                  "message": msg})
        st.write(res.json()["response"])

Writing frontend.py


### Run Streamlit in Colab
Install Streamlit if not already:

In [25]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 34.8 MB/s eta 0:00:00


In [32]:
# Then run your frontend:

!streamlit run frontend.py --server.port 8501 --server.headless true




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.109.137.104:8501

API_URL = "https://giant-jobs-rest.loca.lt"
  Stopping...
